In [54]:
import gurobipy as gp
import pandas as pd
import numpy as np
from matpowercaseframes import CaseFrames

In [82]:
# Gurobipy implementation
case_path = './pglib-opf-21.07/pglib_opf_case300_ieee.m'
ref = CaseFrames(case_path)
# print(f"{ref.bus.PD.sum()/1000:.2f}") # GW
p_max = ref.gen["PMAX"].values    # or ref.gen.PMAX if it's an attribute
p_inf = p_max.max()
p_one = p_max.sum()
alpha_r = 5 * (p_inf / p_one)
print(f"alpha_r = {alpha_r:.6f}  ({alpha_r*100:.2f}%)")

alpha_r = 0.341630  (34.16%)


In [83]:
d_ref = ref.bus["PD"].values   # the “reference” nodal loads (an array of length |N|)


In [ ]:
0.04997

In [ ]:
# 1) sample the global scaling γ ~ U[0.8, 1.2]
gamma = np.random.uniform(0.8, 1.2)
# 2) sample element-wise log-normal noise η_j https://en.wikipedia.org/wiki/Log-normal_distribution
#    so that E[η_j] = 1,  SD[η_j] = 0.05
#    solve μ, σ for the underlying normal:
sigma_ln = np.sqrt(np.log(1 + 0.05**2))      # ≈0.04997
mu_ln    = -0.5 * sigma_ln**2               # ≈ -0.001248
eta = np.random.lognormal(mean=mu_ln,
                          sigma=sigma_ln,
                          size=d_ref.shape)

# 3) element-wise multiply, then scale by γ
d_i = gamma * eta * d_ref

# done!  d_i is your perturbed load vector for instance i
def sample_load(d_ref):
    gamma   = np.random.uniform(0.8, 1.2)
    sigma_ln = np.sqrt(np.log(1 + 0.05**2))
    mu_ln    = -0.5 * sigma_ln**2
    eta      = np.random.lognormal(mu_ln, sigma_ln, size=d_ref.shape)
    return gamma * eta * d_ref

# example: draw 50 000 instances
all_d = np.stack([sample_load(d_ref) for _ in range(50_000)], axis=0)